In [1]:
!pip install -q transformers datasets torch accelerate pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Shuvo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import torch 
import pandas as pd 
from datasets import load_dataset 
from transformers import AutoTokenizer, AutoModelForCausalLM 

In [5]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.to("cpu")
model.eval()

print("Model loaded!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 162.20it/s]


Model loaded!


In [6]:
dataset = load_dataset(
    "cais/mmlu",
    "abstract_algebra"
)

dataset

C:\Users\Shuvo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shuvo\.cache\huggingface\hub\datasets--cais--mmlu. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating dev split: 100%|█████████

DatasetDict({
    test: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 11
    })
    dev: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 5
    })
})

In [7]:
LETTERS = ["A", "B", "C", "D"]

def make_prompt(example):
    question = example["question"]
    choices = example["choices"]

    prompt = f"""Answer the following multiple-choice question.

Question:
{question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Return only one letter: A, B, C, or D.

Answer:"""

    return prompt

In [8]:
print(make_prompt(dataset["test"][0]))

Answer the following multiple-choice question.

Question:
Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.

A. 0
B. 4
C. 2
D. 6

Return only one letter: A, B, C, or D.

Answer:


In [9]:
def get_prediction(example):
    prompt = make_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [11]:
prediction = get_prediction(dataset["test"][0])

print("Model output:", repr(prediction))
print("Correct:", LETTERS[dataset["test"][0]["answer"]])

Model output: 'B\n\nExplanation: The'
Correct: B


In [12]:
import re

def extract_answer(text):
    text = text.upper().strip()

    match = re.search(r"\b([ABCD])\b", text)

    if match:
        return match.group(1)

    return None

In [13]:
print(extract_answer("B"))
print(extract_answer("Answer: C"))
print(extract_answer("The answer is D."))

B
C
D


In [15]:
results = []

NUM_QUESTIONS = min(20, len(dataset["test"]))

test_data = dataset["test"]

for i in range(NUM_QUESTIONS):

    example = test_data[i]

    raw_output = get_prediction(example)
    predicted = extract_answer(raw_output)

    correct = LETTERS[example["answer"]]

    is_correct = predicted == correct

    results.append({
        "question_id": i,
        "prediction": predicted,
        "correct_answer": correct,
        "is_correct": is_correct,
        "raw_output": raw_output
    })

    print(
        f"{i+1}/{NUM_QUESTIONS} | "
        f"Predicted: {predicted} | "
        f"Correct: {correct} | "
        f"{'✓' if is_correct else '✗'}"
    )

1/20 | Predicted: B | Correct: B | ✓
2/20 | Predicted: C | Correct: C | ✓
3/20 | Predicted: C | Correct: D | ✗
4/20 | Predicted: A | Correct: B | ✗
5/20 | Predicted: B | Correct: B | ✓
6/20 | Predicted: A | Correct: A | ✓
7/20 | Predicted: A | Correct: A | ✓
8/20 | Predicted: A | Correct: D | ✗
9/20 | Predicted: B | Correct: B | ✓
10/20 | Predicted: C | Correct: C | ✓
11/20 | Predicted: A | Correct: C | ✗
12/20 | Predicted: D | Correct: C | ✗
13/20 | Predicted: B | Correct: A | ✗
14/20 | Predicted: B | Correct: C | ✗
15/20 | Predicted: B | Correct: C | ✗
16/20 | Predicted: A | Correct: B | ✗
17/20 | Predicted: C | Correct: C | ✓
18/20 | Predicted: C | Correct: C | ✓
19/20 | Predicted: B | Correct: D | ✗
20/20 | Predicted: A | Correct: A | ✓


In [16]:
df = pd.DataFrame(results)

df

,question_id,prediction,correct_answer,is_correct,raw_output
0,0,B,B,True,B\n\nExplanation: The
1,1,C,C,True,C\n\nExplanation: The
2,2,C,D,False,C\n\nExplanation:\nThe
3,3,A,B,False,A\n\nExplanation:\n\nStatement
4,4,B,B,True,B\n\nExplanation:\nThe
5,5,A,A,True,A\n\nExplanation: The
6,6,A,A,True,A\n\nExplanation:\nThe
7,7,A,D,False,A\n\nExplanation:\nStatement
8,8,B,B,True,B\n\nExplanation: The
9,9,C,C,True,C\n\nExplanation:\nThe


In [17]:
accuracy = df["is_correct"].mean()

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 50.00%


In [18]:
correct_count = df["is_correct"].sum()
total_count = len(df)

print("Model:", MODEL_NAME)
print("Dataset: MMLU")
print("Subject: abstract_algebra")
print("Questions:", total_count)
print("Correct:", correct_count)
print("Wrong:", total_count - correct_count)
print(f"Accuracy: {accuracy:.2%}")

Model: Qwen/Qwen3-0.6B
Dataset: MMLU
Subject: abstract_algebra
Questions: 20
Correct: 10
Wrong: 10
Accuracy: 50.00%
